# SimpleSequentialChain 基础


任务太大，输出容易飘

可以使用链式拆解

有点像人为拆解的plan模式

In [7]:
from langchain_classic.chains import LLMChain, SimpleSequentialChain
from langchain_core.prompts import PromptTemplate
from llm_config import build_chat_openai

llm = build_chat_openai(temperature=0, request_timeout=120, max_tokens=1024)

In [8]:
# 任务目标：比较「链式调用」与「单次 Prompt」在输出稳定性上的差异

import time
from langchain_community.callbacks.manager import get_openai_callback


# Step 1: 根据城市推荐代表美食（仅输出名称）
cate_template = """
你是一位美食博主。
根据城市：{city}，推荐一种最有代表性的当地美食。
要求：只输出美食名称，不要解释，不要标点。
"""
cate_prompt = PromptTemplate(template=cate_template, input_variables=["city"])
cate_chain = LLMChain(llm=llm, prompt=cate_prompt)

# Step 2: 基于美食名称写一句简介（仅一句）
intro_template = """
你是一位美食博主。
根据美食：{cate}，写一句简短介绍，突出特点和口感。
要求：只输出1句话，20-35字，不要分点。
"""
intro_prompt = PromptTemplate(template=intro_template, input_variables=["cate"])
intro_chain = LLMChain(llm=llm, prompt=intro_prompt)

# Step 3: 改写为种草文案（只输出文案）
rewrite_template = """
你是一位小红书美食博主。
将下面这句介绍改写成有食欲、自然、像朋友分享的种草文案：
{intro}
要求：
1) 2-3句话；
2) 60-100字；
3) 只输出文案正文，不要标题，不要标签。
"""
rewrite_prompt = PromptTemplate(template=rewrite_template, input_variables=["intro"])
rewrite_chain = LLMChain(llm=llm, prompt=rewrite_prompt)

# Step 4: 基于文案生成 3 个标题（严格三行）
title_template = """
请基于以下文案生成3个吸引人的中文标题：
{content}
要求：
1) 每个标题10-18字；
2) 不要使用emoji；
3) 严格按以下格式输出三行：
标题1：...
标题2：...
标题3：...
"""
title_prompt = PromptTemplate(template=title_template, input_variables=["content"])
title_chain = LLMChain(llm=llm, prompt=title_prompt)

# 1) 链式调用：分步骤逐步完成任务
sequential_chain = SimpleSequentialChain(
    chains=[cate_chain, intro_chain, rewrite_chain, title_chain],
    verbose=True,
)

# 2) 单次 Prompt：一次性交付完整任务
one_shot_template = """
你是一位小红书美食博主，请一次性完成以下任务：
1) 根据城市 {city} 推荐一种最有代表性的当地美食；
2) 用1句话介绍该美食，突出特点和口感（20-35字）；
3) 改写成种草文案（2-3句话，60-100字，语气自然有食欲）；
4) 基于文案生成3个中文标题（每个10-18字，不要emoji）。

严格按以下格式输出：
美食：...
简介：...
文案：...
标题1：...
标题2：...
标题3：...
"""
one_shot_prompt = PromptTemplate(template=one_shot_template, input_variables=["city"])
one_shot_chain = LLMChain(llm=llm, prompt=one_shot_prompt)


def run_with_metrics(run_name, run_fn):
    start = time.perf_counter()
    with get_openai_callback() as cb:
        result = run_fn()
    elapsed = time.perf_counter() - start

    print(f"\n===== {run_name} 结果 =====\n")
    print(result)

    print(f"\n===== {run_name} 调用统计 =====")
    print(f"模型: {getattr(llm, 'model_name', 'unknown')}")
    print(f"耗时: {elapsed:.2f}s")
    print(f"总Tokens: {cb.total_tokens}")
    print(f"输入Tokens: {cb.prompt_tokens}")
    print(f"输出Tokens: {cb.completion_tokens}")
    print(f"成功请求数: {cb.successful_requests}")
    print(f"预估费用(USD): {cb.total_cost:.8f}")


run_with_metrics("链式调用", lambda: sequential_chain.run("成都"))
run_with_metrics("单次 Prompt", lambda: one_shot_chain.run("成都"))


/var/folders/h3/7zc3322s3y1bwclms8c2rjwc0000gn/T/ipykernel_12656/2903520027.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  cate_chain = LLMChain(llm=llm, prompt=cate_prompt)
/var/folders/h3/7zc3322s3y1bwclms8c2rjwc0000gn/T/ipykernel_12656/2903520027.py:54: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  chain_result = sequential_chain.run("成都")




> Entering new SimpleSequentialChain chain...
火锅
沸腾汤底裹挟鲜香麻辣，涮出脆嫩食材，入口醇厚回甘，瞬间唤醒味蕾。
看着红汤咕嘟咕嘟冒泡，那股鲜香麻辣的香气直接往鼻子里钻！随便下点脆嫩的食材，捞起来裹满汤汁咬一口，醇厚里还藏着丝丝回甘，真的瞬间唤醒味蕾，好吃到根本停不下来！
标题1：红汤咕嘟鲜香扑鼻，醇厚回甘一口上瘾
标题2：脆嫩食材裹满红汤，丝丝回甘唤醒味蕾
标题3：鲜香麻辣直抵舌尖，一口入魂根本停不下来

> Finished chain.

===== 链式调用结果 =====

标题1：红汤咕嘟鲜香扑鼻，醇厚回甘一口上瘾
标题2：脆嫩食材裹满红汤，丝丝回甘唤醒味蕾
标题3：鲜香麻辣直抵舌尖，一口入魂根本停不下来

===== 单次 Prompt 结果 =====

美食：成都老火锅
简介：牛油红汤醇厚浓郁，毛肚鸭肠脆嫩弹牙，麻辣鲜香瞬间唤醒味蕾。
文案：钻进成都街巷的老火锅店，看着红油咕嘟冒泡，夹起脆毛肚涮几秒入口，麻辣鲜香瞬间在舌尖炸开。再裹满蒜泥香油碟，解辣提鲜又暖胃，吃完真的让人一口就彻底爱上！
标题1：成都街巷老火锅一口就上头
标题2：红油翻滚的市井麻辣鲜香
标题3：来成都必吃的地道牛油锅


In [12]:
# 议论文写作：链式调用 vs 单次 Prompt（对比示例）
# 主题示例：年轻人是否应该趁早试错

from langchain_classic.chains import LLMChain, SimpleSequentialChain
from langchain_core.prompts import PromptTemplate

# -----------------------------
# 1) 链式调用：分步骤完成议论文
# -----------------------------

# Step 1: 审题 + 立场（输出中心论点）
thesis_template = """
你是一位议论文写作助手。
请根据题目：{topic}
完成：
1) 提炼关键词（不超过3个）
2) 分析核心争议（1句话）
3) 给出明确立场（支持/反对）
4) 写出中心论点（1句话）

输出要求：
- 只输出最后一行：中心论点：...
- 不要输出其他解释。
"""
thesis_prompt = PromptTemplate(template=thesis_template, input_variables=["topic"])
thesis_chain = LLMChain(llm=llm, prompt=thesis_prompt)

# Step 2: 基于中心论点生成3个分论点
arguments_template = """
根据中心论点：{thesis}
请生成3个分论点。

要求：
1) 每个分论点都能直接支撑中心论点
2) 三个分论点角度不同，且有递进关系
3) 每个分论点1句话，避免空话套话

严格按以下格式输出：
分论点1：...
分论点2：...
分论点3：...
"""
arguments_prompt = PromptTemplate(template=arguments_template, input_variables=["thesis"])
arguments_chain = LLMChain(llm=llm, prompt=arguments_prompt)

# Step 3: 为分论点补论据素材
materials_template = """
根据以下分论点内容，补充议论文素材：
{arguments}

要求：
1) 每个分论点补1个道理论据 + 1个事实/例子 + 1句简要分析
2) 例子尽量具体，不要虚构夸张
3) 输出精炼，便于后续生成提纲
"""
materials_prompt = PromptTemplate(template=materials_template, input_variables=["arguments"])
materials_chain = LLMChain(llm=llm, prompt=materials_prompt)

# Step 4: 生成提纲
outline_template = """
请根据以下素材生成议论文提纲：
{materials}

要求：
1) 按“开头-分论点一-分论点二-分论点三-结尾”组织
2) 每部分写出要点（不是全文）
3) 结构清晰，逻辑递进

输出标题：
议论文提纲
"""
outline_prompt = PromptTemplate(template=outline_template, input_variables=["materials"])
outline_chain = LLMChain(llm=llm, prompt=outline_prompt)

# Step 5: 按提纲写正文（输出完整短文）
essay_template = """
请根据以下提纲写一篇议论文正文：
{outline}

要求：
1) 200-300字
2) 包含开头、3个主体段、结尾
3) 观点明确，论证具体，衔接自然
4) 语言正式、简洁、有逻辑
"""
essay_prompt = PromptTemplate(template=essay_template, input_variables=["outline"])
essay_chain = LLMChain(llm=llm, prompt=essay_prompt)

# Step 6: 统一润色
polish_template = """
请润色下面这篇议论文：
{essay}

要求：
1) 保持原观点不变
2) 优化段落衔接与逻辑连贯性
3) 删除重复表达，增强书面感
4) 输出最终稿，不要额外说明
"""
polish_prompt = PromptTemplate(template=polish_template, input_variables=["essay"])
polish_chain = LLMChain(llm=llm, prompt=polish_prompt)

essay_sequential_chain = SimpleSequentialChain(
    chains=[thesis_chain, arguments_chain, materials_chain, outline_chain, essay_chain, polish_chain],
    verbose=True,
)

# essay_chain_result = essay_sequential_chain.run("年轻人是否应该趁早试错")


# -----------------------------
# 2) 单次 Prompt：一次性完成议论文
# -----------------------------

one_shot_essay_template = """
你是一位议论文写作助手，请围绕题目：{topic}
一次性完成以下任务：
1) 给出中心论点（立场明确）
2) 给出3个分论点（角度不同、逻辑递进）
3) 每个分论点补1个道理论据和1个事实例子
4) 先给出简要提纲
5) 再写一篇200-300字议论文（含开头、3个主体段、结尾）

输出格式：
中心论点：...
分论点1：...
分论点2：...
分论点3：...
提纲：...
正文：...
"""
one_shot_essay_prompt = PromptTemplate(template=one_shot_essay_template, input_variables=["topic"])
one_shot_essay_chain = LLMChain(llm=llm, prompt=one_shot_essay_prompt)

# 兜底：如果只单独执行本单元，补齐统计函数
if "run_with_metrics" not in globals():
    import time
    from langchain_community.callbacks.manager import get_openai_callback

    def run_with_metrics(run_name, run_fn):
        start = time.perf_counter()
        with get_openai_callback() as cb:
            result = run_fn()
        elapsed = time.perf_counter() - start

        print(f"\n===== {run_name} 结果 =====\n")
        print(result)

        print(f"\n===== {run_name} 调用统计 =====")
        print(f"模型: {getattr(llm, 'model_name', 'unknown')}")
        print(f"耗时: {elapsed:.2f}s")
        print(f"总Tokens: {cb.total_tokens}")
        print(f"输入Tokens: {cb.prompt_tokens}")
        print(f"输出Tokens: {cb.completion_tokens}")
        print(f"成功请求数: {cb.successful_requests}")
        print(f"预估费用(USD): {cb.total_cost:.8f}")

# 过度分析他人的行为动机，本质上是在瓦解自己的主体性
run_with_metrics("议论文链式调用", lambda: essay_sequential_chain.run("过度分析他人的行为动机，本质上是在瓦解自己的主体性"))
run_with_metrics("议论文单次 Prompt", lambda: one_shot_essay_chain.run("过度分析他人的行为动机，本质上是在瓦解自己的主体性"))



> Entering new SimpleSequentialChain chain...
中心论点：过度分析他人动机不仅会陷入无休止的精神内耗，更会将自我评判权让渡于外界，从而从根本上消解个体的独立意志与主体性。
分论点1：过度拆解他人意图会迫使思维陷入无限假设的闭环，持续消耗认知资源并引发自我怀疑。
分论点2：将自我价值锚定于揣测他人动机之上，会逐步瓦解内在评价体系，使个体主动交出定义对错的权力。
分论点3：当行为逻辑完全受制于对外部意图的迎合与防备时，个体将彻底丧失自主决策的根基，最终消解独立意志与主体性。
以下素材已按“理论+实例+分析”结构精炼编排，可直接嵌入议论文提纲：

**【分论点1】**
- **道理论据**：心理学“认知带宽”理论指出，工作记忆容量有限，过度推演未明信息会触发“反刍思维”，持续挤占本用于理性分析与执行的认知资源，最终导致决策疲劳与自我效能感骤降。
- **事实/例子**：职场中典型的“已读未回”内耗。某员工因领导未及时回复工作请示，连续数日推演“是否方案被否”“是否遭边缘化”，在反复假设中延误项目节点，并陷入“我能力不足”的自我否定。
- **简要分析**：该案例印证了意图拆解如何形成认知闭环：当大脑将算力消耗于无解的动机推演时，执行资源被抽空，自我怀疑便成为认知超载的必然副产品。

**【分论点2】**
- **道理论据**：社会学“价值外化”机制表明，当个体将自我认同的坐标系完全锚定于揣测他人动机时，内在的价值标尺会被“他者凝视”取代，导致个体主动让渡对是非对错的定义权。
- **事实/例子**：内容创作者的“数据迎合”困境。部分创作者长期以“受众是否喜欢”为唯一标准，不断揣测平台偏好与读者意图并修改内容，最终丧失个人表达风格，甚至对“何为优质作品”失去独立判断，完全依赖评论风向与流量反馈。
- **简要分析**：此例揭示了价值锚点外移的代价：当评判标准从“内在准则”滑向“动机揣测”，内在评价体系随之瓦解，对错定义权彻底沦为外部意图的附庸。

**【分论点3】**
- **道理论据**：组织行为学中的“防御性决策”现象指出，当行为逻辑完全受制于对外部意图的迎合与防备时，内在动机将被“外部动机挤出效应”削弱，行动退化为条件反射式的风险规避，自主决策的根基由此崩塌。
- **事实/例子**：企业中层管理者的“揣摩式执行”。某管